# Top-5 Finish Prediction Model
Trains a logistic regression model to predict whether a team will finish in the Premier League top 5, using only data available up to a given matchday cutoff (15, 20, or 30). Trained on 25 historical seasons (2000/01–2024/25), tested against the completed 2025/26 season.

**Method note:** momentum feature uses the exact win/draw/loss candlestick formula from the original momentum-tracking notebooks — win +1, loss −1, draw decays 50% back toward zero based on prior trend direction.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import pickle

pd.set_option('display.max_columns', None)
print("Libraries loaded.")

Libraries loaded.


## 1. Load training data (25 seasons, 2000/01–2024/25)

In [2]:
# Place epl_final.csv in the same folder as this notebook
train_raw = pd.read_csv('epl_final.csv')
train_raw['MatchDate'] = pd.to_datetime(train_raw['MatchDate'])
print(train_raw['Season'].nunique(), 'seasons |', train_raw.shape[0], 'matches')
train_raw.head()

25 seasons | 9380 matches


,Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals,FullTimeResult,HalfTimeHomeGoals,HalfTimeAwayGoals,HalfTimeResult,HomeShots,AwayShots,HomeShotsOnTarget,AwayShotsOnTarget,HomeCorners,AwayCorners,HomeFouls,AwayFouls,HomeYellowCards,AwayYellowCards,HomeRedCards,AwayRedCards
0,2000/01,2000-08-19,Charlton,Man City,4,0,H,2,0,H,17,8,14,4,6,6,13,12,1,2,0,0
1,2000/01,2000-08-19,Chelsea,West Ham,4,2,H,1,0,H,17,12,10,5,7,7,19,14,1,2,0,0
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3,A,1,1,D,6,16,3,9,8,4,15,21,5,3,1,0
3,2000/01,2000-08-19,Derby,Southampton,2,2,D,1,2,A,6,13,4,6,5,8,11,13,1,1,0,0
4,2000/01,2000-08-19,Leeds,Everton,2,0,H,2,0,H,17,12,8,6,6,4,21,20,1,3,0,0


## 2. Team name normalization
The 2025/26 test data uses full club names (e.g. 'Manchester City'); the training data uses short forms (e.g. 'Man City'). This map keeps them consistent.

In [3]:
NAME_MAP = {
    'Manchester City': 'Man City',
    'Manchester United': 'Man United',
    'Newcastle United': 'Newcastle',
    'Tottenham Hotspur': 'Tottenham',
    'Wolverhampton Wanderers': 'Wolves',
    'Brighton & Hove Albion': 'Brighton',
    'West Ham United': 'West Ham',
    'Nottingham Forest': "Nott'm Forest",
    'Leeds United': 'Leeds',
    'Chelsea FC': 'Chelsea',
}

def normalize(df, home_col='HomeTeam', away_col='AwayTeam'):
    df = df.copy()
    df[home_col] = df[home_col].replace(NAME_MAP)
    df[away_col] = df[away_col].replace(NAME_MAP)
    return df

## 3. Momentum + feature functions
`build_team_sequence` reproduces the original momentum candlestick formula (win/draw/loss → running level, EMA-5 smoothed) for one team across a season.
`features_at_cutoff` computes points, goal difference, W/D/L, PPG, recent-5-match form, and EMA-5 momentum using only the first `cutoff` matches — no data from beyond that point is used.

In [4]:
def build_team_sequence(df, team, date_col='MatchDate'):
    rows = []
    for _, r in df.iterrows():
        if r['HomeTeam'] == team:
            gf, ga = r['FullTimeHomeGoals'], r['FullTimeAwayGoals']
            res = 1 if r['FullTimeResult']=='H' else (0 if r['FullTimeResult']=='D' else -1)
        elif r['AwayTeam'] == team:
            gf, ga = r['FullTimeAwayGoals'], r['FullTimeHomeGoals']
            res = 1 if r['FullTimeResult']=='A' else (0 if r['FullTimeResult']=='D' else -1)
        else:
            continue
        rows.append({date_col: r[date_col], 'GF': gf, 'GA': ga, 'Result': res})
    seq = pd.DataFrame(rows).sort_values(date_col).reset_index(drop=True)

    # momentum candle Close value — same formula as the original notebooks
    current_level = 0.0
    last_color = None
    closes = []
    for _, row in seq.iterrows():
        open_price = current_level
        result = row['Result']
        if result == 1:
            close_price = open_price + 1.0
            last_color = 'green'
        elif result == -1:
            close_price = open_price - 1.0
            last_color = 'red'
        else:
            if last_color == 'green':
                close_price = max(open_price - open_price*0.5, 0)
            elif last_color == 'red':
                close_price = min(open_price + (0-open_price)*0.5, 0)
            else:
                close_price = 0.0
                last_color = 'gray'
        closes.append(close_price)
        current_level = close_price
    seq['MomentumClose'] = closes
    seq['EMA5'] = seq['MomentumClose'].ewm(span=5, adjust=False).mean()
    seq['Points'] = seq['Result'].map({1:3, 0:1, -1:0})
    return seq


def features_at_cutoff(seq, cutoff):
    if len(seq) < cutoff:
        return None
    sub = seq.iloc[:cutoff]
    pts = sub['Points'].sum()
    gf = sub['GF'].sum(); ga = sub['GA'].sum()
    w = (sub['Result']==1).sum(); d = (sub['Result']==0).sum(); l = (sub['Result']==-1).sum()
    return {
        'Points': pts, 'GD': gf-ga, 'GF': gf, 'GA': ga,
        'W': w, 'D': d, 'L': l, 'PPG': pts/cutoff,
        'RecentFormPts5': sub.tail(5)['Points'].sum(),
        'EMA5': sub['EMA5'].iloc[-1]
    }

## 4. Build labeled training data at each checkpoint

In [5]:
CUTOFFS = [15, 20, 30]
FEATURES = ['Points','GD','GF','GA','W','D','L','PPG','RecentFormPts5','EMA5']

all_rows = {c: [] for c in CUTOFFS}
seasons = sorted(train_raw['Season'].unique())

for season in seasons:
    sdf = train_raw[train_raw['Season']==season].copy()
    teams = sorted(set(sdf['HomeTeam']) | set(sdf['AwayTeam']))

    final_table = []
    for t in teams:
        seq = build_team_sequence(sdf, t)
        final_table.append((t, seq['Points'].sum(), seq['GF'].sum()-seq['GA'].sum(), seq['GF'].sum()))
    tb = pd.DataFrame(final_table, columns=['Team','Pts','GD','GF']).sort_values(
        ['Pts','GD','GF'], ascending=False).reset_index(drop=True)
    top5_teams = set(tb.iloc[:5]['Team'])

    for t in teams:
        seq = build_team_sequence(sdf, t)
        for cutoff in CUTOFFS:
            feats = features_at_cutoff(seq, cutoff)
            if feats is None:
                continue
            feats['Season'] = season
            feats['Team'] = t
            feats['Top5'] = 1 if t in top5_teams else 0
            all_rows[cutoff].append(feats)

train_data = {c: pd.DataFrame(all_rows[c]) for c in CUTOFFS}
for c in CUTOFFS:
    print(c, train_data[c].shape, 'positive rate:', round(train_data[c]['Top5'].mean(),3))

15 (500, 13) positive rate: 0.25
20 (500, 13) positive rate: 0.25
30 (500, 13) positive rate: 0.25


## 5. Train + validate (leave-one-season-out) vs. naive points baseline
The naive baseline simply takes the top 5 teams by points at the cutoff. This is the bar our model needs to clear — per the Pielke (2015) finding that most published football prediction models fail to beat naive baselines.

In [8]:
models = {}
scalers = {}
results_summary = {}

for cutoff in CUTOFFS:
    df = train_data[cutoff]
    seasons_list = df['Season'].unique()
    oof_preds = np.zeros(len(df))
    naive_correct = 0; naive_total = 0

    for test_season in seasons_list:
        train_mask = df['Season'] != test_season
        test_mask = df['Season'] == test_season

        scaler = StandardScaler().fit(df.loc[train_mask, FEATURES].values)
        X_train = scaler.transform(df.loc[train_mask, FEATURES].values)
        X_test = scaler.transform(df.loc[test_mask, FEATURES].values)

        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, df.loc[train_mask, 'Top5'].values)
        oof_preds[test_mask.values] = clf.predict_proba(X_test)[:,1]

        sdf = df.loc[test_mask]
        naive_top5 = set(sdf.sort_values('Points', ascending=False).head(5)['Team'])
        actual_top5 = set(sdf[sdf['Top5']==1]['Team'])
        naive_correct += len(naive_top5 & actual_top5); naive_total += 5

    df = df.copy()
    df['oof_pred'] = oof_preds
    model_correct = 0; model_total = 0
    for season in seasons_list:
        sdf = df[df['Season']==season]
        pred_top5 = set(sdf.sort_values('oof_pred', ascending=False).head(5)['Team'])
        actual_top5 = set(sdf[sdf['Top5']==1]['Team'])
        model_correct += len(pred_top5 & actual_top5); model_total += 5

    auc = roc_auc_score(df['Top5'].values, oof_preds)
    results_summary[cutoff] = {
        'AUC': round(auc,3),
        'model_hit_rate': round(model_correct/model_total,3),
        'naive_hit_rate': round(naive_correct/naive_total,3)
    }

    # fit final model on ALL seasons for use on new data
    scaler_final = StandardScaler().fit(df[FEATURES].values)
    clf_final = LogisticRegression(max_iter=1000)
    clf_final.fit(scaler_final.transform(df[FEATURES].values), df['Top5'].values)
    models[cutoff] = clf_final
    scalers[cutoff] = scaler_final

    print(f"Matchday {cutoff}: AUC={results_summary[cutoff]['AUC']} | "
          f"model hit-rate={results_summary[cutoff]['model_hit_rate']} | "
          f"naive hit-rate={results_summary[cutoff]['naive_hit_rate']}")
    print("  coefficients:", dict(zip(FEATURES, clf_final.coef_[0].round(3))))

Matchday 15: AUC=0.961 | model hit-rate=0.824 | naive hit-rate=0.824
  coefficients: {'Points': 0.725, 'GD': 0.515, 'GF': 0.66, 'GA': -0.15, 'W': 0.66, 'D': 0.08, 'L': -0.756, 'PPG': 0.725, 'RecentFormPts5': -0.143, 'EMA5': 0.17}
Matchday 20: AUC=0.979 | model hit-rate=0.92 | naive hit-rate=0.888
  coefficients: {'Points': 0.73, 'GD': 0.922, 'GF': 0.98, 'GA': -0.572, 'W': 0.716, 'D': -0.155, 'L': -0.671, 'PPG': 0.73, 'RecentFormPts5': -0.261, 'EMA5': 0.376}
Matchday 30: AUC=0.986 | model hit-rate=0.904 | naive hit-rate=0.904
  coefficients: {'Points': 0.986, 'GD': 0.814, 'GF': 0.678, 'GA': -0.771, 'W': 0.993, 'D': -0.337, 'L': -0.881, 'PPG': 0.986, 'RecentFormPts5': 0.197, 'EMA5': -0.506}


## 6. Test against the completed 2025/26 season
Load the dated, matchday-ordered 2025/26 results (from openfootball) and apply the trained models at each checkpoint. The actual outcome is known and held separately below — it is never fed into feature computation.

In [9]:
# Place epl_2025_26.csv in the same folder as this notebook
test_raw = pd.read_csv('epl_2025_26.csv')
test_raw = normalize(test_raw)
test_raw = test_raw.reset_index().rename(columns={'index':'MatchOrder'})

ACTUAL_TOP5 = {'Arsenal','Man City','Man United','Aston Villa','Liverpool'}  # ground truth, held out

teams_2526 = sorted(set(test_raw['HomeTeam']) | set(test_raw['AwayTeam']))

for cutoff in CUTOFFS:
    rows = []
    for t in teams_2526:
        seq = build_team_sequence(test_raw, t, date_col='MatchOrder')
        feats = features_at_cutoff(seq, cutoff)
        if feats is None:
            continue
        feats['Team'] = t
        rows.append(feats)
    fdf = pd.DataFrame(rows)
    Xs = scalers[cutoff].transform(fdf[FEATURES].values)
    fdf['ModelProb'] = models[cutoff].predict_proba(Xs)[:,1]

    predicted_top5 = set(fdf.sort_values('ModelProb', ascending=False).head(5)['Team'])
    naive_top5 = set(fdf.sort_values('Points', ascending=False).head(5)['Team'])

    print(f"\n=== Matchday {cutoff} ===")
    print('Model predicted top 5:', predicted_top5, '-> correct:', len(predicted_top5 & ACTUAL_TOP5), '/5')
    print('Naive points top 5:   ', naive_top5, '-> correct:', len(naive_top5 & ACTUAL_TOP5), '/5')
    display(fdf.sort_values('ModelProb', ascending=False)[['Team','Points','GD','EMA5','ModelProb']].head(8))


=== Matchday 15 ===
Model predicted top 5: {'Arsenal', 'Aston Villa', 'Chelsea', 'Man City', 'Crystal Palace'} -> correct: 3 /5
Naive points top 5:    {'Arsenal', 'Aston Villa', 'Chelsea', 'Man City', 'Crystal Palace'} -> correct: 3 /5


,Team,Points,GD,EMA5,ModelProb
0,Arsenal,33,19,2.789221,0.975346
12,Man City,31,19,4.547656,0.952753
1,Aston Villa,30,7,5.705342,0.772985
7,Crystal Palace,26,8,1.012743,0.468976
6,Chelsea,25,10,0.826756,0.468700
13,Man United,25,4,0.761079,0.380659
4,Brighton,23,4,1.037349,0.246545
17,Tottenham,22,7,-0.494933,0.193871



=== Matchday 20 ===
Model predicted top 5: {'Arsenal', 'Aston Villa', 'Chelsea', 'Man City', 'Liverpool'} -> correct: 4 /5
Naive points top 5:    {'Arsenal', 'Aston Villa', 'Chelsea', 'Man City', 'Liverpool'} -> correct: 4 /5


,Team,Points,GD,EMA5,ModelProb
0,Arsenal,48,26,5.367305,0.998615
12,Man City,42,26,4.933230,0.994593
1,Aston Villa,42,9,9.415930,0.954741
6,Chelsea,31,11,0.052803,0.326602
11,Liverpool,34,4,1.509750,0.266156
13,Man United,31,4,0.334310,0.164121
3,Brentford,30,4,0.760588,0.094066
14,Newcastle,29,4,0.220728,0.078156



=== Matchday 30 ===
Model predicted top 5: {'Arsenal', 'Aston Villa', 'Chelsea', 'Man United', 'Man City'} -> correct: 4 /5
Naive points top 5:    {'Arsenal', 'Aston Villa', 'Man United', 'Man City', 'Liverpool'} -> correct: 5 /5


,Team,Points,GD,EMA5,ModelProb
0,Arsenal,67,37,2.567568,0.998933
12,Man City,61,32,2.287581,0.990759
13,Man United,54,13,3.386767,0.593456
6,Chelsea,48,18,0.377619,0.384780
1,Aston Villa,51,3,-0.634635,0.326480
11,Liverpool,49,9,0.713716,0.323116
3,Brentford,45,4,0.798996,0.076370
14,Newcastle,42,0,-1.568477,0.041620


## 7. Save models for reuse (e.g. on the live 2026/27 season)

In [10]:
with open('final_models.pkl', 'wb') as f:
    pickle.dump({'models': models, 'scalers': scalers, 'features': FEATURES}, f)
print('Saved final_models.pkl')

Saved final_models.pkl
